# 04. LoRA 모델 평가 + 최종 경제성 보고서 (L4, 약 1시간 / 약 2유닛)
드라이브 `cfinetune/gguf/`에 있는 GGUF를 02와 똑같은 방법으로 평가하고, 모든 결과로 표와 그래프를 만듭니다.

In [ ]:
# 저장소 받기 + 패키지 설치
BRANCH = "ccr-a463bea6-bpbyrh"   # main에 합친 뒤에는 "main"
REPO = "github.com/b1bibig/Cfinetune"
import os
try:  # 저장소가 비공개면 코랩 왼쪽 열쇠(보안 비밀)에 GH_TOKEN 을 넣어 두세요
    from google.colab import userdata
    _tok = userdata.get("GH_TOKEN")
    REPO_URL = f"https://{_tok}@{REPO}"
except Exception:
    REPO_URL = f"https://{REPO}"
if not os.path.exists("/content/Cfinetune"):
    !git clone -q -b {BRANCH} {REPO_URL} /content/Cfinetune
%cd /content/Cfinetune
!git pull -q
%pip install -q -r requirements-colab.txt

In [ ]:
# 구글 드라이브: 결과(results/)를 세션이 끊겨도 남도록 저장
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, sys
DRIVE = "/content/drive/MyDrive/cfinetune"
os.makedirs(DRIVE, exist_ok=True)

def pull_results():
    if os.path.exists(f"{DRIVE}/results"):
        shutil.copytree(f"{DRIVE}/results", "results", dirs_exist_ok=True)

def push_results():
    shutil.copytree("results", f"{DRIVE}/results", dirs_exist_ok=True)
    print("드라이브에 결과 저장함")

pull_results()
sys.path.insert(0, "scripts"); sys.path.insert(0, "src")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU 메모리 {mem:.0f}GB")
assert mem >= 20, "런타임 > 런타임 유형 변경에서 L4 를 고르세요"

In [ ]:
!bash scripts/build_llama_cpp.sh /content/llama.cpp {DRIVE}/cache

In [ ]:
import glob
for g in sorted(glob.glob(f"{DRIVE}/gguf/*.gguf")):
    name = os.path.basename(g)[:-5]
    if os.path.exists(f"results/eval/{name}.json") and os.path.exists(f"results/bench/{name}.json"):
        print("건너뜀:", name); continue
    local = f"/content/{name}.gguf"
    shutil.copy(g, local)            # 드라이브에서 바로 읽으면 느려서 로컬로 복사
    !python scripts/05_eval.py --gguf {local} --name {name}
    !python scripts/06_bench.py --gguf {local} --name {name}
    os.remove(local)
    push_results()

In [ ]:
!python scripts/07_econ.py --measured-gpu "L4 (Colab)"
push_results()
from IPython.display import Markdown, Image, display
display(Markdown(open("results/report.md").read()))
for f in ("fig_size_vs_korean.png", "fig_energy.png", "fig_expert_overlap.png"):
    if os.path.exists(f"results/{f}"): display(Image(f"results/{f}"))